# smoke test (overfit diagnostic)

Train on a fixed 32-example subset, evaluate on those **same** 32, and require token-sim to jump hard.

**Pass condition:** token-sim delta > 0.3 on the subset (and ideally the model emits EOS / stops
on its own — see the diagnostic cells). If it passes, the loop is sound and you move to the real
training notebook.

In [ ]:
# === config ===
import os, csv, json, time, math, difflib, random
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from torch.utils.data import DataLoader

MODEL    = "Qwen/Qwen2.5-Coder-0.5B"
REPO     = "notoh/exebench_llvm_o3"
REVISION = "4d26c13bf8473067db611fce495ce4dd27639c0a"   # pinned cleaned corpus

OUT_DIR  = Path("runs") / (MODEL.split("/")[-1] + "_smoke")
MAX_LEN  = 7600            # >= dataset cap (7500) + template tokens

# overfit-diagnostic knobs: high LR + many steps, we WANT fast memorization of a tiny set
OVERFIT_N    = 32          # fixed subset; train == eval
LR           = 1e-4
GRAD_ACCUM   = 4
TARGET_STEPS = 150
WARMUP       = 10
BATCH        = 1
EVAL_N       = 20          # eval examples (<= OVERFIT_N)
PROMPT_TMPL  = "Compile the following C to LLVM IR at -O3:\n{c}\n---\n"

OUT_DIR.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# === load tokenizer, model, data (pinned revision) ===
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

dtype = torch.bfloat16 if device == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype)
model.to(device)
model.gradient_checkpointing_enable()

train_ds = load_dataset(REPO, revision=REVISION)["train"]
print(f"train pool: {len(train_ds)}")

In [ ]:
# === fixed overfit subset: train == eval on the SAME 32 examples ===
idx     = random.Random(0).sample(range(len(train_ds)), min(OVERFIT_N, len(train_ds)))
fit_ds  = train_ds.select(idx)
eval_ds = fit_ds
print(f"overfit subset: {len(fit_ds)} fixed examples (seed 0)")

In [ ]:
# === generation budget: GEN_MAX_NEW must cover the IR we ask the model to produce ===
import numpy as np
tgt_lens = [len(tok(r["o3_ir"], add_special_tokens=False)["input_ids"]) for r in eval_ds]
p50, p95, p99 = (int(np.percentile(tgt_lens, q)) for q in (50, 95, 99))
print(f"o3_ir target tokens  p50={p50}  p95={p95}  p99={p99}  max={max(tgt_lens)}")
GEN_MAX_NEW = max(1024, int(math.ceil(p99 / 256) * 256))
over = sum(l > GEN_MAX_NEW for l in tgt_lens)
print(f"GEN_MAX_NEW = {GEN_MAX_NEW}  ({over}/{len(tgt_lens)} targets exceed it)")

In [ ]:
# === collation + completion-only loss masking ===
EOS_ID = tok.eos_token_id

def build_example(rec):
    prompt = PROMPT_TMPL.format(c=rec["c_source"])
    p_ids  = tok(prompt, add_special_tokens=False)["input_ids"]
    t_ids  = tok(rec["o3_ir"], add_special_tokens=False)["input_ids"] + [EOS_ID]  # append EOS id
    input_ids = (p_ids + t_ids)[:MAX_LEN]
    labels    = ([-100]*len(p_ids) + t_ids)[:MAX_LEN]   # mask prompt, grade IR (+EOS)
    return input_ids, labels

def collate(batch):
    rows = [build_example(r) for r in batch]
    maxlen = max(len(x[0]) for x in rows)
    pad = tok.pad_token_id
    input_ids, labels, attn = [], [], []
    for ids, lab in rows:
        n = maxlen - len(ids)
        input_ids.append(ids + [pad]*n)
        labels.append(lab + [-100]*n)
        attn.append([1]*len(ids) + [0]*n)
    return {"input_ids": torch.tensor(input_ids),
            "labels": torch.tensor(labels),
            "attention_mask": torch.tensor(attn)}

# sanity: graded tokens > 0; flag MAX_LEN truncation (would cut the EOS)
for idx in random.sample(range(len(fit_ds)), min(5, len(fit_ds))):
    ids, lab = build_example(fit_ds[idx])
    graded = sum(1 for x in lab if x != -100)
    assert graded > 0, f"row {idx}: zero graded tokens (empty/truncated IR?)"
    print(f"row {idx:>4}  graded {graded:>5}"
          f"  {'[TRUNCATED at MAX_LEN -> EOS lost]' if len(ids) >= MAX_LEN else ''}")

In [ ]:
# === eval helpers (used for BOTH baseline and trained, same model object) ===
def normalize(s):
    return "\n".join(l.rstrip() for l in s.strip().splitlines())

def token_sim(a, b):
    ta = tok(a, add_special_tokens=False)["input_ids"]
    tb = tok(b, add_special_tokens=False)["input_ids"]
    return difflib.SequenceMatcher(None, ta, tb).ratio()

def run_eval(model, label, eval_ds, n, eval_batch=8):
    was_training = model.training
    model.eval()
    prev_cache, prev_side = model.config.use_cache, tok.padding_side
    model.config.use_cache = True
    tok.padding_side = "left"                 # correct decoder-only batched generation
    n = min(n, len(eval_ds))
    exact, sims = 0, []
    for start in range(0, n, eval_batch):
        recs    = [eval_ds[k] for k in range(start, min(start + eval_batch, n))]
        prompts = [PROMPT_TMPL.format(c=r["c_source"]) for r in recs]
        enc = tok(prompts, add_special_tokens=False, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=GEN_MAX_NEW,
                                 do_sample=False, pad_token_id=tok.pad_token_id)
        new = gen[:, enc["input_ids"].shape[1]:]
        for j, r in enumerate(recs):
            pred = tok.decode(new[j], skip_special_tokens=True)
            if normalize(pred) == normalize(r["o3_ir"]):
                exact += 1
            sims.append(token_sim(pred, r["o3_ir"]))
    tok.padding_side, model.config.use_cache = prev_side, prev_cache
    if was_training:
        model.train()
    ef, ms = exact / n, sum(sims) / len(sims)
    print(f"[{label}] eval on {n} (same train set):  exact {exact}/{n} ({ef:.0%})  token-sim {ms:.3f}")
    return ef, ms

In [ ]:
# === BASELINE eval (UNTRAINED) — before any optimizer step touches weights ===
base_exact, base_sim = run_eval(model, "untrained baseline", eval_ds, EVAL_N)

In [ ]:
# === train loop (overfit the 32) ===
train_loader    = DataLoader(fit_ds, batch_size=BATCH, shuffle=True, collate_fn=collate)
num_batches     = len(train_loader)
steps_per_epoch = math.ceil(num_batches / GRAD_ACCUM)
total_steps     = TARGET_STEPS
print(f"num_batches={num_batches}  steps/epoch={steps_per_epoch}  total_steps={total_steps}")

opt   = torch.optim.AdamW(model.parameters(), lr=LR)
sched = get_cosine_schedule_with_warmup(opt, WARMUP, max(total_steps, 1))

logf   = open(OUT_DIR / "train_log.csv", "w", newline="")
writer = csv.writer(logf); writer.writerow(["step", "loss", "lr", "secs"])

model.config.use_cache = False          # required with gradient checkpointing during training
model.train()
step, t0, window_loss = 0, time.time(), 0.0
opt.zero_grad()
stop = False
while not stop:
    for i, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        win_start = (i // GRAD_ACCUM) * GRAD_ACCUM
        win_size  = min(GRAD_ACCUM, num_batches - win_start)
        loss = model(**batch).loss / win_size
        loss.backward()
        window_loss += loss.item()
        if ((i + 1) % GRAD_ACCUM == 0) or ((i + 1) == num_batches):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            step += 1
            secs = round(time.time() - t0, 1)
            print(f"step {step}/{total_steps}  loss {window_loss:.4f}  lr {sched.get_last_lr()[0]:.2e}  {secs}s")
            writer.writerow([step, window_loss, sched.get_last_lr()[0], secs]); logf.flush()
            window_loss = 0.0
            if step >= total_steps:
                stop = True; break
logf.close()

ckpt = OUT_DIR / "checkpoint"
model.save_pretrained(ckpt); tok.save_pretrained(ckpt)
print("saved checkpoint ->", ckpt)

In [ ]:
# === TRAINED eval + verdict ===
trained_exact, trained_sim = run_eval(model, "trained", eval_ds, EVAL_N)

print("\n=== smoke verdict ===")
print(f"  token sim:   {base_sim:.3f} -> {trained_sim:.3f}  (delta {trained_sim - base_sim:+.3f})")
print(f"  exact match: {base_exact:.0%} -> {trained_exact:.0%}")
if (trained_sim - base_sim) > 0.3:
    print("PASS — the loop learns. Move to the real training notebook.")
else:
    print("FAIL — loop is NOT learning. Do NOT scale. Check, in order:")
    print("  1) train_log.csv loss curve: did it fall? flat=no learning, nan/0=all labels masked")
    print("  2) one printed example (prompt / pred / ref) to see what generation produces")
    print("  3) graded-token counts in the collate sanity cell")

In [ ]:
# === diagnostic: EOS plumbing ===
print("eos_token    :", repr(tok.eos_token))
print("eos_token_id :", tok.eos_token_id)
print("pad_token_id :", tok.pad_token_id, "(pad==eos?", tok.pad_token_id == tok.eos_token_id, ")")

rec = next(r for r in fit_ds if len(build_example(r)[0]) < MAX_LEN)   # non-truncated row
ids, lab = build_example(rec)
print("\nfinal input id :", ids[-1], "  == eos_token_id?", ids[-1] == tok.eos_token_id)
print("final label    :", lab[-1], "  graded (not -100)?", lab[-1] != -100)

In [ ]:
# === diagnostic: does the trained model emit EOS / stop on its own? ===
rec = eval_ds[0]
ids = tok(PROMPT_TMPL.format(c=rec["c_source"]), add_special_tokens=False, return_tensors="pt").to(device)
model.eval(); model.config.use_cache = True
out = model.generate(**ids, max_new_tokens=GEN_MAX_NEW, do_sample=False, pad_token_id=tok.pad_token_id)
new = out[0][ids["input_ids"].shape[1]:]
has_eos = (new == tok.eos_token_id).any().item()
print("output length     :", len(new), "/ ceiling", GEN_MAX_NEW)
print("eos id in output? :", has_eos)
print("first eos position:", (new == tok.eos_token_id).nonzero().min().item() if has_eos else "none")